In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Đọc file
data_dir = "/kaggle/input/en2vi-dataset/"
en_sents = open(data_dir + "en_sents", "r", encoding="utf-8").read().splitlines()
vi_sents = open(data_dir + "vi_sents", "r", encoding="utf-8").read().splitlines()

# Tạo DataFrame và chia train/test
df = pd.DataFrame({"en": en_sents, "vi": vi_sents})
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Chuyển DataFrame thành Dataset của Hugging Face
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Xóa cột __index__ nếu có
train_dataset = train_dataset.remove_columns(['__index_level_0__'] if '__index_level_0__' in train_dataset.column_names else [])
test_dataset = test_dataset.remove_columns(['__index_level_0__'] if '__index_level_0__' in test_dataset.column_names else [])

print("Train dataset sample:", train_df.head())
print("Test dataset sample:", test_df.head())

Train dataset sample:                                                        en  \
230184                    Don't leave your things behind.   
197873                                 Come on everybody.   
33271             You don't know where Tom works, do you?   
87823                                That helped me a lot   
5371    I heard about the accident for the first time ...   

                                                       vi  
230184                      đừng bỏ lại những thứ của bạn  
197873                                   cố lên mọi người  
33271              bạn không biết tom làm việc ở đâu chứ?  
87823                       Điều đó đã giúp tôi rất nhiều  
5371    tôi đã nghe về vụ tai nạn lần đầu tiên vào ngà...  
Test dataset sample:                                                      en  \
52530                   I lost my camera the other day.   
31525                  I'm not very good with children.   
167843                      Don't you worry about tha

In [2]:
import os
from transformers import AutoTokenizer
import torch
from datasets import Dataset

# Tắt song song hóa tokenizer để tránh cảnh báo deadlock
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Kiểm tra GPU
print(f"GPU khả dụng: {torch.cuda.is_available()}")
print(f"Thiết bị GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Không có'}")

# Tải tokenizer
model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Hàm tiền xử lý với đệm và chuyển đổi tensor đúng cách
def preprocess(examples):
    # Thêm tiền tố cho từng chuỗi trong danh sách
    src_texts = ["translate English to Vietnamese: " + en for en in examples["en"]]
    tgt_texts = examples["vi"]
    # Mã hóa đầu vào với đệm và cắt ngắn
    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    # Mã hóa nhãn với đệm và cắt ngắn
    labels = tokenizer(
        tgt_texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    model_inputs["labels"] = labels["input_ids"]
    # Loại bỏ kích thước batch (squeeze) để đảm bảo tensor đúng định dạng
    model_inputs["input_ids"] = model_inputs["input_ids"].squeeze()
    model_inputs["attention_mask"] = model_inputs["attention_mask"].squeeze()
    model_inputs["labels"] = model_inputs["labels"].squeeze()
    return model_inputs

# Áp dụng tiền xử lý
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=["en", "vi"])
tokenized_test = test_dataset.map(preprocess, batched=True, remove_columns=["en", "vi"])

# Lưu tập dữ liệu đã mã hóa
tokenized_train.save_to_disk("./tokenized_train")
tokenized_test.save_to_disk("./tokenized_test")

# Kiểm tra mẫu dữ liệu đã mã hóa
print("Tokenized train sample:", tokenized_train[0])
print("Tokenized test sample:", tokenized_test[0])

GPU khả dụng: True
Thiết bị GPU: Tesla P100-PCIE-16GB


tokenizer_config.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

Map:   0%|          | 0/228681 [00:00<?, ? examples/s]

Map:   0%|          | 0/25409 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/228681 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25409 [00:00<?, ? examples/s]

Tokenized train sample: {'input_ids': [31286, 2722, 14071, 1904, 3844, 5071, 35862, 2368, 35906, 35787, 4223, 5325, 27659, 868, 3, 35807, 15, 1970, 2319, 35792, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [4916, 783, 265, 204, 739, 54, 1113, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [3]:

import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback, get_linear_schedule_with_warmup
import torch
import gc
from datasets import load_from_disk

# Cấu hình môi trường
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU khả dụng: {torch.cuda.is_available()}")
print(f"Thiết bị: {device}")

# Xóa bộ nhớ GPU
gc.collect()
torch.cuda.empty_cache()

# Tải mô hình và tokenizer
model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Bật gradient checkpointing để tiết kiệm bộ nhớ
model.gradient_checkpointing_enable()
model.to(device)

# Tải và giảm kích thước dữ liệu
tokenized_train = load_from_disk("./tokenized_train").select(range(5000))  # Tăng lên 48,000 mẫu
tokenized_test = load_from_disk("./tokenized_test").select(range(1000))

# Tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./vit5-checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    per_device_train_batch_size=4,  # Giảm để tránh OOM
    per_device_eval_batch_size=2,
    num_train_epochs=5,  # Giảm để tránh overfitting
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    fp16_full_eval=True,
    dataloader_num_workers=1,  # Giảm để tránh bottleneck
    gradient_accumulation_steps=2,  # Batch size hiệu quả = 8
)

# Tạo optimizer và scheduler
num_training_steps = len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs
num_warmup_steps = int(0.1 * num_training_steps)  

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# Huấn luyện
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    optimizers=(optimizer, scheduler)
)

trainer.train()

# Lưu mô hình
output_model_dir = "./vit5-model-final"
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Mô hình và tokenizer đã được lưu vào {output_model_dir}")

# Giải phóng bộ nhớ
gc.collect()
torch.cuda.empty_cache()

2025-04-24 22:11:40.558748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745532700.752581      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745532700.808549      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


GPU khả dụng: True
Thiết bị: cuda


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,0.144400,0.122277
2,0.095200,0.098549
3,0.058400,0.096734
4,0.040700,0.100656
5,0.027100,0.108239


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Mô hình và tokenizer đã được lưu vào ./vit5-model-final


In [4]:
# Cài đặt thư viện cần thiết
!pip install rouge-score sacrebleu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=9887a57fd0d8d93c018dd7270d6b071e40e1d468191beff72719be5f851cfa86
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [5]:
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from datasets import load_from_disk
from rouge_score import rouge_scorer
import sacrebleu
import numpy as np

# Cấu hình môi trường
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU khả dụng: {torch.cuda.is_available()}")
print(f"Thiết bị: {device}")

# Tải mô hình và tokenizer
model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained("./vit5-model-final")

# Chuyển mô hình sang GPU nếu có
model.to(device)
model.eval()

# Tải tập dữ liệu kiểm tra đã mã hóa
test_dataset = load_from_disk("./tokenized_test").select(range(1000))  # Bạn có thể thay đổi phạm vi này tùy theo tập kiểm tra

# Hàm dự đoán
def generate_predictions(inputs):
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    return outputs

# Dự đoán trên tập kiểm tra
predictions = []
references = []
for example in test_dataset:
    inputs = {
        "input_ids": torch.tensor(example["input_ids"]).unsqueeze(0),
        "attention_mask": torch.tensor(example["attention_mask"]).unsqueeze(0)
    }
    pred_ids = generate_predictions(inputs)
    pred_text = tokenizer.decode(pred_ids[0], skip_special_tokens=True)
    ref_text = tokenizer.decode(example["labels"], skip_special_tokens=True)
    predictions.append(pred_text)
    references.append(ref_text)

# Tính ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)
    for key in rouge_scores:
        rouge_scores[key].append(scores[key].fmeasure)

# Tính trung bình ROUGE
rouge_avg = {key: np.mean(values) for key, values in rouge_scores.items()}

# Tính BLEU
bleu = sacrebleu.corpus_bleu(predictions, [references])

# In kết quả
print("ROUGE Scores:")
print(f"  ROUGE-1: {rouge_avg['rouge1']:.4f}")
print(f"  ROUGE-2: {rouge_avg['rouge2']:.4f}")
print(f"  ROUGE-L: {rouge_avg['rougeL']:.4f}")
print("BLEU Score:")
print(f"  BLEU: {bleu.score:.4f}")

# Lưu kết quả (tùy chọn)
with open("evaluation_results.txt", "w") as f:
    f.write("ROUGE Scores:\n")
    f.write(f"  ROUGE-1: {rouge_avg['rouge1']:.4f}\n")
    f.write(f"  ROUGE-2: {rouge_avg['rouge2']:.4f}\n")
    f.write(f"  ROUGE-L: {rouge_avg['rougeL']:.4f}\n")
    f.write("BLEU Score:\n")
    f.write(f"  BLEU: {bleu.score:.4f}\n")


GPU khả dụng: True
Thiết bị: cuda
ROUGE Scores:
  ROUGE-1: 0.7234
  ROUGE-2: 0.5691
  ROUGE-L: 0.6992
BLEU Score:
  BLEU: 41.5404


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Thiết lập thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tải tokenizer và mô hình đã fine-tuned
model_dir = "./vit5-model-final"  # Đường dẫn thư mục chứa mô hình đã huấn luyện
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)
model.eval()

# Câu tiếng Anh cần dịch
src_sentence = "My family was not poor , and myself , I had never experienced hunger"

# Thêm prefix nếu mô hình yêu cầu (ví dụ vit5-base cần "translate English to Vietnamese: ")
input_text = f"translate English to Vietnamese: {src_sentence}"
inputs = tokenizer(input_text, return_tensors="pt", padding=True).to(device)

# Dịch
with torch.no_grad():
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Giải mã kết quả
translated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Câu gốc:", src_sentence)
print("Dịch:", translated_text)

Câu gốc: My family was not poor , and myself , I had never experienced hunger
Dịch: gia đình tôi không nghèo, và tôi không bao giờ cảm thấy cô đơn
